<a href="https://colab.research.google.com/github/Lemonade-exe/python_research_2026/blob/main/STEP_ResearchQuestion_AlgorithmsV9.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Meeting Notes
- 2/28/2026 Saturday
- Read and understand kNN algorithm
- run the with small sizes of parameters then in new colab notebooks extend the number of parameters gradually.



# Meeting Notes
- 2/21/2026 Saturday
- Read and understand kNN algorithm
- convert your code into nested for loops
- save your results in a dataframe




In [ ]:
# Libraries needed for data preparation
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
import yfinance as yf
import pandas as pd
import numpy as np
import csv
import requests

# Lag data / input
'''
def lag_func(data, name, lag):
    df_lag = pd.DataFrame(data[name])
    for i in range(1, lag+1):
        df_lag[f'lag_{i}'] = df_lag[name].shift(i)
        df_lag.dropna(inplace=True)
    return df_lag
'''

# Close data / output
def lag_func(data, name, lag):
    df_lag = data[['Date', name]].copy()

    for i in range(1, lag + 1):
        df_lag[f'lag_{i}'] = df_lag[name].shift(i)

    df_lag.dropna(inplace=True)
    return df_lag

# Data Preparation for Google
# Time Range
                # 1 month, 6 month, 1 year, 5 year, 10 year
startValues = ["2005-1-1", "2006-1-1", "2007-1-1", "2009-1-1", "2015-1-1"]
endValues = ["2005-2-1", "2006-7-1", "2008-1-1", "2014-1-1", "2025-1-1"]

START = "2010-1-1"
END = "2024-12-31"

# Creates/declare Dataframe
df = pd.DataFrame()

# Initialize the dataframe
df['Google'] = yf.Ticker('GOOG').history(start=START, end=END).Close

  # Removes hour
df.reset_index(inplace=True)
df['Date'] = [i.date() for i in df.Date]
df['Up/Down'] = (df['Google'].shift(-1) > df['Google']).astype(int)
# return dataframe
lag_func(df,'Google',5).head().round(4) ## Missing Date

In [ ]:
#lagged df
df_lagged = lag_func(df, 'Google', 5)

#lagged data
df_lagged['Up/Down'] = df['Up/Down'].iloc[df_lagged.index]
df_lagged.dropna(inplace=True)
x, y = df_lagged[[f'lag_{i}' for i in range(1, 6)]], df_lagged['Up/Down']

# checks the amount of data
print(x.shape)
print(y.shape)

In [ ]:
scaler = StandardScaler()
x_scaled = scaler.fit_transform(x)

#splitting data
split = int(len(x_scaled) * 0.8)

# declaring the x_train, y_train, x_test, y_test
x_train, x_test = x_scaled[:split], x_scaled[split:]
y_train, y_test = y.iloc[:split], y.iloc[split:]

# KNN

In [ ]:
# KNN Classifier
from sklearn.neighbors import KNeighborsClassifier

# DataFrame declaration and initialization
KNN_df = pd.DataFrame(columns = ['N_neighbors', 'Weights', 'Algorithm', 'Leaf Size', 'Power', 'Train Score', 'Test Score'])

# Parameter Set
KNN_algorithm = {"auto", "ball_tree", "kd_tree", "brute"}
KNN_weights = {None, "uniform", "distance"}

# Model (n_neighbors)
def KNN_func(trainX, trainY, testX, testY):
  for NNeighbors in range(1,11):
    for Weights in KNN_weights:
      for Algorithm in KNN_algorithm:
        for LeafSize in range(1,11):
          for Power in np.arange(1,11): # (42,45) (53,55) (55,57)
            knn = KNeighborsClassifier(n_neighbors = NNeighbors, weights = Weights, algorithm = Algorithm, leaf_size = LeafSize, p = Power)
            knn.fit(trainX, trainY)

            # Prediction
            # knn.predict(x_train[:5]), knn.predict(x_test[:5])

            # Score
            trainScore = knn.score(trainX, trainY)
            testScore = knn.score(testX, testY)

            # Update DataFrame
            KNN_df.loc[len(KNN_df)] = [NNeighbors, Weights, Algorithm, LeafSize, Power, trainScore, testScore]

            if(testScore >= 0.6):
              return #NNeighbors, Weights, Algorithm, LeafSize, Power, trainScore, testScore

KNN_func(x_train,y_train,x_test,y_test)

In [ ]:
KNN_df.round(2).sort_values('Test Score', ascending=False)

In [ ]:
# KNN Classifier
from sklearn.neighbors import KNeighborsClassifier

# DataFrame declaration and initialization
KNN_df = pd.DataFrame(columns = ['N_neighbors', 'Weights', 'Algorithm', 'Leaf Size', 'Power', 'Train Score', 'Test Score'])

# Parameter Set
KNN_algorithm = {"auto", "ball_tree", "kd_tree", "brute"}
KNN_weights = {None, "uniform", "distance"}


# Model (n_neighbors)
for NNeighbors in range(1,11):
  for Weights in KNN_weights:
    for Algorithm in KNN_algorithm:
      for LeafSize in range(1,11):
        for Power in np.arange(1,11): # (42,45) (53,55) (55,57)
          knn = KNeighborsClassifier(n_neighbors = NNeighbors, weights = Weights, algorithm = Algorithm, leaf_size = LeafSize, p = Power)
          knn.fit(x_train, y_train)

          # Prediction
          knn.predict(x_train[:5]), knn.predict(x_test[:5])

          # Score
          trainScore = knn.score(x_train, y_train)
          testScore = knn.score(x_test, y_test)

          # Update DataFrame
          KNN_df.loc[len(KNN_df)] = [NNeighbors, Weights, Algorithm, LeafSize, Power, trainScore, testScore]

          if(testScore >= 0.6):
            key = False
            print(NNeighbors, Weights, Algorithm, LeafSize, Power, trainScore, testScore)



In [ ]:
KNN_df.round(2).sort_values('Test Score', ascending=False)

# Decision Tree

In [ ]:
# Decision Tree
from sklearn.tree import DecisionTreeClassifier

# DataFrame declaration and initialization
DT_df = pd.DataFrame(columns = ['Max Depth', 'Minimum Samples Split', 'Minimum Samples Leaf', 'Max Features', 'Random State', 'Maximum Leaf Nodes', 'Train Score', 'Test Score'])

# Model (max_depth)
for MaxDepth in range(1,5):
  for MinSamplesSplit in range(2,5):
    for MinSamplesLeaf in range(2,5):
      for MaxFeatures in range(1,5):
        for RandomState in range(1,5):
          for MaxLeafNodes in range(2,5):
            # Model
            dt = DecisionTreeClassifier(max_depth = MaxDepth, min_samples_split = MinSamplesSplit, min_samples_leaf = MinSamplesLeaf, max_features = MaxFeatures, random_state = RandomState, max_leaf_nodes = MaxLeafNodes)
            dt.fit(x_train, y_train)

            # Prediction
            (dt.predict(x_train[:5]) >= 0.5).astype(int)
            (dt.predict(x_test[:5]) >= 0.5).astype(int)

            # Score
            trainScore = dt.score(x_train,y_train)
            testScore = dt.score(x_test,y_test)

            # Update DataFrame
            DT_df.loc[len(DT_df)] = [MaxDepth, MinSamplesSplit, MinSamplesLeaf, MaxFeatures, RandomState, MaxLeafNodes, trainScore, testScore]


In [ ]:
DT_df.round(2).sort_values('Test Score', ascending = False)

# Random Forest

In [ ]:
# Random Forest
from sklearn.ensemble import RandomForestClassifier

# DataFrame declaration and initialization
RF_df = pd.DataFrame(columns = ['N_Estimators', 'Random State', 'Minimum Samples Split', 'Verbose', 'n_jobs', 'Train Score', 'Test Score'])

for NEstimators in range (1,10):
  for RandomState in range(1,10):
    for MinSamplesSplit in range(2,10):
      for ver in range(1,10):
        for nJobs in range (1,10):
          # Model
          rf = RandomForestClassifier(n_estimators = NEstimators, random_state = RandomState, min_samples_split = MinSamplesSplit, verbose = ver, n_jobs = nJobs)
          rf.fit(x_train,y_train)

          # Prediction
          (rf.predict(x_train[:5]) >= 0.5).astype(int)
          (rf.predict(x_test[:5]) >= 0.5).astype(int)

          # Score
          trainScore = rf.score(x_train,y_train)
          testScore = rf.score(x_test,y_test)

          # Update DataFrame
          RF_df.loc[len(RF_df)] = [NEstimators, RandomState, MinSamplesSplit, ver, nJobs, trainScore, testScore]



In [ ]:
RF_df.round(2).sort_values('Test Score', ascending = False)

# XGBoost

In [ ]:
# XGBoost Classifier
from xgboost import XGBClassifier

# DataFrame declaration and initialization
XGB_df = pd.DataFrame(columns = ['N_Estimators', 'Max Depth', 'Learning Rate', 'Max Leaves', 'n_jobs', 'Train Score', 'Test Score'])

for NEstimators in range(1,5):
  for MaxDepth in range(1,5):
    for LearningRate in range(1,5):
      for leaves in range (1,5):
        for nJobs in range (1,5):
          # Model
          xgb = XGBClassifier(n_estimators = NEstimators, max_depth = MaxDepth, learning_rate = LearningRate, max_leaves = leaves, n_jobs = nJobs)
          xgb.fit(x_train, y_train)

          # Prediction
          (xgb.predict(x_train[:5]) >= 0.5).astype(int)
          (xgb.predict(x_test[:5]) >= 0.5).astype(int)

          # Score
          trainScore = xgb.score(x_train,y_train)
          testScore = xgb.score(x_test,y_test)

          # Update Dataframe
          XGB_df.loc[len(XGB_df)] = [NEstimators, MaxDepth, LearningRate, leaves, nJobs, trainScore, testScore]

In [ ]:
XGB_df.round(2).sort_values('Test Score', ascending = False)

# Logistic Regression

In [ ]:
# Logistic Regression
from sklearn.linear_model import LogisticRegression

# DataFrame Declaration and initialization
Log_df = pd.DataFrame(columns = ['Inverse of Regularization Strength', 'Elastic-Net Mixing','Max Iteration', 'Verbose','Train Score', 'Test Score'])

# Parameter Set
Log_penalty = {"l1", "l2", "elasticnet", None}

# Model
for Penalty in Log_penalty:
  for InverseRegStren in np.arange(1,3):
    for ElasticNetMix in np.arange(0,1):
      for MaxIter in range(100,1000,300):
        for Verbose in range(0,3):
          # Model
          log = LogisticRegression(penalty = Penalty, C = InverseRegStren,l1_ratio = ElasticNetMix, max_iter = MaxIter, verbose = Verbose)
          log.fit(x_train, y_train)

          # Prediction
          (log.predict(x_train[:5]) >= 0.5).astype(int)
          (log.predict(x_test[:5]) >= 0.5).astype(int)

          # Score
          trainScore = log.score(x_train, y_train)
          testScore = log.score(x_test, y_test)

          # Update Dataframe
          Log_df.loc[len(Log_df)] = [Penalty, InverseRegStren, ElasticNetMix, MaxIter, Verbose, trainScore, testScore]

In [ ]:
Log_df.round(2).sort_values('Test Score', ascending=False)